EA refund calculation - Download from CST 

[Preparation]
1. Usage download from CST - Invoice Usage csv file
2. sub_list csv file

**For sub_list.csv file you can filter by subId or subname.

1) Filter SubID: create csv file with first column input by 'S|ubID'
2) Filter SubName: create csv file with first column input by 'SubName'


In [3]:
import pandas as pd
import dask.dataframe as dd

pd.options.display.float_format = '{:.7f}'.format
pd.set_option('mode.chained_assignment', None)

#Filter by subName please Input "Y" after subName=
#Filter by subID please Input "N" after subName=
#no filter use please Input "S" after subName=
subName = "S"

##[Filter by Sub ID]
# Input your sublist file name in '' from read_csv('') including .csv file format 
# And delete "#" infront of subInfo= 
# Then add "#" infront of second subInfo= 
subInfo=pd.read_csv('subID.csv')['SubID'].values.tolist() 

##[Filter by Sub Name]
# Input your sublist file name in '' from read_csv('') including .csv file format 
# And delete "#" infront of subInfo= 
# Then add "#" infront of first subInfo= 
#subInfo=pd.read_csv('mysubName.csv')['name'].values.tolist() 

# Input your usage download file name after url= including .csv file format
url = 'upfront.csv' 

usage = dd.read_csv(url, usecols=['SubscriptionId', 'SubscriptionName','PricingModel','ChargeType','Cost','Date','ReservationId','ReservationName', 'benefitId']
                    ,low_memory=False
                   ,dtype={'ReservationId': 'object',
                           'ReservationName': 'object',
                           'benefitId': 'object'
                          })
subList = []

if(subName == "N"):
    for a in subInfo:
    if a not in subList:
        subList.append(a)
    filtered = usage.loc[usage["SubscriptionId"].isin(subList)]
    pd_usage = filtered.compute()
elif (subName == "Y"):
    for a in subInfo:
    if a not in subList:
        subList.append(a)
    filtered = usage.loc[usage["SubscriptionName"].isin(subList)]
    pd_usage = filtered.compute()
else:
    pd_usage = usage.compute()


print("Usage Filtering Done!!! Please proceed cell below!")

FileNotFoundError: [Errno 2] No such file or directory: 'subID.csv'

In [17]:
####Total Usage######################################################################################################
summary = pd.pivot_table(pd_usage, index=["SubscriptionId", "SubscriptionName"], values=["Cost"], aggfunc="sum", margins=True, margins_name="Total")
result = pd.DataFrame(summary.to_records())
display(summary)

Cost
SubscriptionId                       SubscriptionName     PricingModel ChargeType                  
04145959-f91c-4db5-8a1b-86494879cd7a sko-dedicated        OnDemand     Usage         343600.4170478
                                                          Reservation  Purchase      281630.0000000
                                                                       Usage              0.0000000
07cb8198-79b6-4c37-b1c6-6f231d23a518 ski-erp              OnDemand     Usage       23829844.6425244
2bd37708-3e7d-4997-8db5-f2a459c37978 skl-dedicated        OnDemand     Usage         254371.3780492
2f604512-e874-4b39-b08d-3b326ae71b59 ske-platform-dev     OnDemand     Usage       10576506.8235637
3324f15f-ca29-4c33-a26d-3a0961319b0f ski-dedicated        OnDemand     Usage        1141454.0008364
                                                          Reservation  Purchase      337956.0000000
                                                                       Usage              0.0000000
377c6bf8-d4ea-46b6-8ada-9b2d94ae187a ski-datagov          OnDemand     Usage       12807004.2115865
4abf50f9-7aca-472c-8672-b085bb8f5750 ski-platform-devops  OnDemand     Usage        6275717.5803758
4bac66d3-3dc3-4a5d-b6f1-906d592932c5 ski-platform-common  Reservation  Purchase     1829420.0000000
5d53de27-ac3f-4915-bd5f-397a213053f3 ski-global-sslvpn-ea OnDemand     Usage         397229.4036153
65e91fd0-b496-4332-9cbc-14ad02e70dcf ski-landingzone-ea   OnDemand     Usage       10481289.9221436
                                                          Reservation  Purchase      362136.0000000
                                                                       Usage              0.0000000
6aa830bc-aae5-42de-b4ea-c783cd5ea030 ske-platform-cv-ea   OnDemand     Usage        6749729.2211714
76376155-668c-4619-b582-05de035cd92b ski-dt-testbed       OnDemand     Usage        1020485.3885775
78c52490-981f-47c1-a8e8-9f6537261971 ski-dr-ea            OnDemand     Usage        3702433.0831508
                                                          Reservation  Purchase     1235802.0000000
                                                                       Usage              0.0000000
8b5a97da-b5ce-4edb-b1f6-26301f0214af ski-iactest          OnDemand     Usage          74612.7218493
b4e77a00-3cd6-4605-9e06-c7e841ed176a ski-vdi              OnDemand     Usage        1926774.7154213
b56fc911-3028-4e0a-a836-2fe06a07eadd clx-platform-common  OnDemand     Usage       14449995.3180076
                                                          Reservation  Purchase     4864656.0000000
                                                                       Usage              0.0000000
d01545a6-8c88-418d-94d8-fb271a1a73f3 sko-baas             OnDemand     Usage       10840075.6352933
dd4efaab-0205-4d42-a3ac-7e00a466f9b8 skgc-widtus          OnDemand     Usage         163784.5346730
ea29350e-a785-476b-8533-59002a2cb353 ski-dataanalysis     OnDemand     Usage        8493486.4954152
ee3ad727-8b03-4783-94d7-85033d532c49 skipc-dedicated      OnDemand     Usage          96196.2610566
f2ae9408-3500-45d0-b5be-5cc9a26e2075 ski-common-ea        OnDemand     Usage       11234270.3587760
                                                          Reservation  Purchase     4175162.0000000
                                                                       Usage              0.0000000
f92911a9-3375-46cf-9898-61d28a72fc94 skeo-dedicated       OnDemand     Usage        4209794.9200299
fc652697-9c2c-40de-ad34-d8c330569d49 ske-platform-muffin  OnDemand     Usage        8504517.2671097
Total                                                                             150659936.3002744

In [19]:
####Calculate Usage refund###########################################################################################
TotalAmount = 0
refund = 0
usageFlag = True
sort_usage = pd_usage[pd_usage.PricingModel =='OnDemand']
if(sort_usage.empty):
    print("#### No Usage ####")
    usageFlag = False
else:
    usage = pd.pivot_table(sort_usage, index=["SubscriptionId"], values=["Cost"], aggfunc="sum", margins=True, margins_name="Total")
    usage.reset_index(inplace=True)
    TotalAmount = usage.tail(n=1)['Cost']
    refund = TotalAmount*0.1
    usage.loc[len(usage.index)] = ['Refund(Total*10%)',refund.values[0]]
    display(usage)

,SubscriptionId,Cost
0,04145959-f91c-4db5-8a1b-86494879cd7a,343600.4170478
1,07cb8198-79b6-4c37-b1c6-6f231d23a518,23829844.6425244
2,2bd37708-3e7d-4997-8db5-f2a459c37978,254371.3780492
3,2f604512-e874-4b39-b08d-3b326ae71b59,10576506.8235637
4,3324f15f-ca29-4c33-a26d-3a0961319b0f,1141454.0008364
5,377c6bf8-d4ea-46b6-8ada-9b2d94ae187a,12807004.2115865
6,4abf50f9-7aca-472c-8672-b085bb8f5750,6275717.5803758
7,5d53de27-ac3f-4915-bd5f-397a213053f3,397229.4036153
8,65e91fd0-b496-4332-9cbc-14ad02e70dcf,10481289.9221436
9,6aa830bc-aae5-42de-b4ea-c783cd5ea030,6749729.2211714


In [21]:
##Download
file_name = 'refundResult.xlsx'
with pd.ExcelWriter(file_name) as writer:
#    total.to_excel(writer, sheet_name="Total Refund Amount", startrow=1, startcol=1)
    if(usageFlag):
        usage.to_excel(writer, sheet_name="Usage Refund Calc", startrow=1, startcol=1, index=False)
#    if(riFlag):
#        riTable.to_excel(writer, sheet_name="RI Refund Calc", startrow=1, startcol=1, index=False)
    result.to_excel(writer, sheet_name="raw Usage Summary", startrow=1, startcol=1, index=False)
display("DONE!! " + file_name + " file")

'DONE!! Go to https://20.214.188.193:8000/tree/Refund refundResult.xlsx file'